<a href="https://colab.research.google.com/github/jothiprakasam/llm/blob/main/llm/course/pipelines.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

behind the pipelines

In [16]:
from transformers import AutoModelForSequenceClassification  , AutoTokenizer
checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [18]:
raw_inputs = ["hi there","Hello my dear friends"]
inputs = tokenizer(raw_inputs,padding=True,return_tensors="pt")
print(inputs)
outputs = model(**inputs)
# Now that we use AutoModelForSequenceClassification, we access the 'logits' attribute
print("Logits:", outputs.logits)

{'input_ids': tensor([[ 101, 7632, 2045,  102,    0,    0],
        [ 101, 7592, 2026, 6203, 2814,  102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 0, 0],
        [1, 1, 1, 1, 1, 1]])}
Logits: tensor([[-3.4778,  3.6936],
        [-3.6908,  3.9704]], grad_fn=<AddmmBackward0>)


multiple sequence

In [19]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [22]:
sequence = "I've been waiting for a HuggingFace course my whole life."

tokens = tokenizer.tokenize(sequence)
ids = tokenizer.convert_tokens_to_ids(tokens)
batch_ids=[ids,ids]
input_ids = torch.tensor(batch_ids)
print("Input IDs:", input_ids)

output = model(input_ids)
print("Logits:", output.logits)

Input IDs: tensor([[ 1045,  1005,  2310,  2042,  3403,  2005,  1037, 17662, 12172,  2607,
          2026,  2878,  2166,  1012],
        [ 1045,  1005,  2310,  2042,  3403,  2005,  1037, 17662, 12172,  2607,
          2026,  2878,  2166,  1012]])
Logits: tensor([[-2.7276,  2.8789],
        [-2.7276,  2.8789]], grad_fn=<AddmmBackward0>)




```
# batch_ids = [ids,ids]
```
This above process is called batching which sends multipe sends sentence at once. cause tensor accepts multiple sentences.


sequnc_ids and padding_ids

In [23]:
import torch

In [25]:
seq1 = [[200,200]]
seq2 = [[200,200,200]]
print(model(torch.tensor(seq1)).logits)
print(model(torch.tensor(seq2)).logits)

tensor([[ 0.5803, -0.4125]], grad_fn=<AddmmBackward0>)
tensor([[ 1.5694, -1.3895]], grad_fn=<AddmmBackward0>)


In [34]:
# let me pad and check the values
# batch it for model input
padding_ids = tokenizer.pad_token_id
print(padding_ids)
seq3 = [[200,200,padding_ids]]
print(model(torch.tensor(seq3)).logits)

0
tensor([[ 1.3374, -1.2163]], grad_fn=<AddmmBackward0>)


we can see the logits are different from those [200,200,200] and [200,200,pads]. both are same but we have different logits.
so let me add attention mask.

In [35]:
attention_mask = [[1,1,0]]
print(model(torch.tensor(seq3),attention_mask=torch.tensor(attention_mask)).logits)

tensor([[ 0.5803, -0.4125]], grad_fn=<AddmmBackward0>)


Now, we have got same tensors logits.